In [1]:
import gensim.downloader as api
wv=api.load("word2vec-google-news-300")

In [2]:
wv.similarity(w1="great",w2="good")

0.729151

In [3]:
# to see the vector of a particular word
wv["good"]

array([ 0.04052734,  0.0625    , -0.01745605,  0.07861328,  0.03271484,
       -0.01263428,  0.00964355,  0.12353516, -0.02148438,  0.15234375,
       -0.05834961, -0.10644531,  0.02124023,  0.13574219, -0.13183594,
        0.17675781,  0.27148438,  0.13769531, -0.17382812, -0.14160156,
       -0.03076172,  0.19628906, -0.03295898,  0.125     ,  0.25390625,
        0.12695312, -0.15234375,  0.03198242,  0.01135254, -0.01361084,
       -0.12890625,  0.01019287,  0.23925781, -0.08447266,  0.140625  ,
        0.13085938, -0.04516602,  0.06494141,  0.02539062,  0.05615234,
        0.24609375, -0.20507812,  0.23632812, -0.00860596, -0.02294922,
        0.05078125,  0.10644531, -0.03564453,  0.08740234, -0.05712891,
        0.08496094,  0.23535156, -0.10107422, -0.03564453, -0.04736328,
        0.04736328, -0.14550781, -0.10986328,  0.14746094, -0.23242188,
       -0.07275391,  0.19628906, -0.37890625, -0.07226562,  0.04833984,
        0.11914062,  0.06103516, -0.12109375, -0.27929688,  0.05

In [4]:
wv["good"].shape

(300,)

In [5]:
import pandas as pd

In [6]:
df=pd.read_csv("Fake_Real_Data.csv")
df.head()

,Text,label
0,Top Trump Surrogate BRUTALLY Stabs Him In The...,Fake
1,U.S. conservative leader optimistic of common ...,Real
2,"Trump proposes U.S. tax overhaul, stirs concer...",Real
3,Court Forces Ohio To Allow Millions Of Illega...,Fake
4,Democrats say Trump agrees to work on immigrat...,Real


In [7]:
df.label.value_counts()
# to check any class imbalance

label
Fake    5000
Real    4900
Name: count, dtype: int64

In [8]:
df['label_num']=df['label'].map({"Fake":0,"Real":1})
df.head()

,Text,label,label_num
0,Top Trump Surrogate BRUTALLY Stabs Him In The...,Fake,0
1,U.S. conservative leader optimistic of common ...,Real,1
2,"Trump proposes U.S. tax overhaul, stirs concer...",Real,1
3,Court Forces Ohio To Allow Millions Of Illega...,Fake,0
4,Democrats say Trump agrees to work on immigrat...,Real,1


In [9]:
import spacy
nlp=spacy.load("en_core_web_lg")


def preprocess_and_vectorize(text):
    doc=nlp(text)
    
    filtered_tokens=[]
    for token in doc:
        if token.is_stop or token.is_punct:
            continue
        filtered_tokens.append(token.lemma_)
    return wv.get_mean_vector(filtered_tokens)

In [10]:
preprocess_and_vectorize("ok, don't worry,come and meet me tomorrow").shape

(300,)

In [12]:
df["vector"]=df['Text'].apply(lambda text:preprocess_and_vectorize(text))
df.head()
# it takes atleast 15 minutes to run

,Text,label,vector
0,Top Trump Surrogate BRUTALLY Stabs Him In The...,Fake,"[0.008657642, 0.019024342, -0.011917442, 0.032..."
1,U.S. conservative leader optimistic of common ...,Real,"[0.010864096, 0.007960429, 0.0011915653, 0.014..."
2,"Trump proposes U.S. tax overhaul, stirs concer...",Real,"[0.018134918, 0.0062743523, -0.005872244, 0.03..."
3,Court Forces Ohio To Allow Millions Of Illega...,Fake,"[0.01255197, 0.012613623, 5.9780963e-05, 0.021..."
4,Democrats say Trump agrees to work on immigrat...,Real,"[-0.0019059887, 0.011889367, 0.0035395357, 0.0..."


In [16]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(df.vector.values,df.label_num,test_size=0.2,random_state=1,stratify=df.label_num)

In [17]:
import numpy as np

In [19]:
x_train.shape

(7920,)

In [20]:
x_test.shape

(1980,)

In [21]:
# converting x_train,x_test as 2d array using numpy stack function
x_train=np.stack(x_train)
x_test=np.stack(x_test)

In [22]:
x_train.shape

(7920, 300)

In [23]:
x_test.shape

(1980, 300)

In [25]:
from sklearn.ensemble import GradientBoostingClassifier

model=GradientBoostingClassifier()
model.fit(x_train,y_train)

GradientBoostingClassifier()

In [29]:
y_test[:5]

1883    0
7747    1
5898    0
3335    1
8173    1
Name: label_num, dtype: int64

In [30]:
y_pred=model.predict(x_test)
y_pred[:5]

array([0, 1, 0, 1, 1], dtype=int64)

In [28]:
from sklearn.metrics import classification_report
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.99      0.98      0.98      1000
           1       0.98      0.99      0.98       980

    accuracy                           0.98      1980
   macro avg       0.98      0.98      0.98      1980
weighted avg       0.98      0.98      0.98      1980

